# Notebook 2: The Export-Side Model (Rice)

**What this notebook does:** it asks a simple question, can we tell, in advance, that a country is about to stop selling its rice abroad? It explains what signals we chose to look at, why we chose them, tests them properly, and shows exactly which country's pattern held up and which didn't, with real reasons why.

**Files to upload, in order:** `events_clean_final.csv` (from Notebook 1), `Comtrade_2014-2001.csv`, `Comtrade_2014-2025.csv` (Rice trade), `production.csv`


## Setup

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LeaveOneOut
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from google.colab import files


In [ ]:
uploaded = files.upload()

In [ ]:
events = pd.read_csv("events_clean_final.csv")
events["Start_Year"] = pd.to_datetime(events["Start_Date"]).dt.year
rice_events = events[events["CommodityClass_Name"]=="Rice"].copy()
rice_relevant_countries = sorted(rice_events["Country_Name"].unique())
print("Countries with a real Rice restriction history:", rice_relevant_countries)


---
## Step 1: Why these three signals, Price, Quantity, and Production?

Before testing anything, here is the actual reasoning behind each one, and what we expected each to tell us.

**Price:** when a government restricts exports, it almost always says it is doing this to protect its own people from rising food prices. So the price a country's rice sells for abroad is the closest thing we have to what actually triggers the decision. If price is rising fast before a restriction, that is the government's own stated reason showing up in the data.

**Quantity:** a country selling unusually large amounts abroad right up until it suddenly stops is a real, visible behavior pattern. It also captures something price alone cannot, whether the country was in a selling rush before pulling back.

**Production:** a bad harvest is often the root cause behind a country's price problem in the first place. If a government is worried about a shortage at home, that shortage often starts with a drop in how much was actually grown that year. This signal lets us catch a cause that happens even earlier than a price spike.

**Why all three together, not just one:** each one captures a different part of the story, buyer-facing price, the country's own selling behavior, and the underlying supply. Using all three lets the model see the fuller picture, not just one slice of it.

**One honest limit, stated upfront:** the price we use here is the price paid by foreign buyers, since that is what trade data actually records. It is a close stand-in for the domestic price pain a government is really reacting to, but it is not the exact same number.


## Step 2: Build the trade and production data

In [ ]:
uploaded = files.upload()

In [ ]:
c1 = pd.read_csv("Comtrade_2014-2001.csv", index_col=False)
c2 = pd.read_csv("Comtrade_2014-2025.csv", index_col=False)
comtrade = pd.concat([c1,c2], ignore_index=True)
comtrade = comtrade[comtrade["flowDesc"]=="Export"]
trade = comtrade.groupby(["reporterDesc","refYear"], as_index=False).agg(
    Export_Quantity_t=("qty", lambda x: x.sum()/1000), Export_Value_USD=("primaryValue","sum"))
trade = trade.rename(columns={"reporterDesc":"Country","refYear":"Year"})
trade["Country"] = trade["Country"].replace({"China":"China, mainland"})
trade = trade[trade["Country"].isin(rice_relevant_countries)].copy()

zero_check = trade[trade["Export_Quantity_t"]==0]
print("Rows with zero quantity but real value (a data error to fix):")
print(zero_check)


**What we found:** Vietnam, 2012, shows 0 tonnes exported alongside 3.68 billion dollars in value. That is impossible for one of the world's largest rice exporters, it is a reporting gap, not a real zero. We treat it as missing, not as an actual zero, so it does not silently break the math later.


In [ ]:
trade["Export_Quantity_t"] = trade["Export_Quantity_t"].replace(0, np.nan)
trade["Unit_Price_USD_per_t"] = trade["Export_Value_USD"] / trade["Export_Quantity_t"]
trade = trade.replace([np.inf,-np.inf], np.nan)


In [ ]:
uploaded = files.upload()

In [ ]:
production_raw = pd.read_csv("production.csv")
production = production_raw[production_raw["Element"]=="Production"].copy()
production = production[production["Item"]=="Rice"][["Area","Year","Value"]].rename(columns={"Area":"Country","Value":"Production_t"})
production["Country"] = production["Country"].replace({"China":"China, mainland"})

full = pd.merge(trade, production, on=["Country","Year"], how="inner")
full = full.sort_values(["Country","Year"]).reset_index(drop=True)
print("Merged dataset:", full.shape)

full["Prev_Year"] = full.groupby("Country")["Year"].shift(1)
full["YoY_Reliable"] = (full["Year"]-full["Prev_Year"])==1
full.loc[full["Prev_Year"].isna(),"YoY_Reliable"]=False
full["Price_YoY_pct"] = full.groupby("Country")["Unit_Price_USD_per_t"].pct_change(fill_method=None)*100
full["Quantity_YoY_pct"] = full.groupby("Country")["Export_Quantity_t"].pct_change(fill_method=None)*100
full["Production_YoY_pct"] = full.groupby("Country")["Production_t"].pct_change(fill_method=None)*100
for col in ["Price_YoY_pct","Quantity_YoY_pct","Production_YoY_pct"]:
    full.loc[~full["YoY_Reliable"], col] = None

ban_lookup = set(zip(rice_events["Country_Name"], rice_events["Start_Year"]))
full["Restriction_Next_Year"] = full.apply(lambda r: 1 if (r["Country"], r["Year"]+1) in ban_lookup else 0, axis=1)
reliable = full[full["YoY_Reliable"]==True].copy()
print("Reliable rows, real year-over-year changes only:", len(reliable))


---
## Step 3: Test the hypothesis, all countries together first

**The hypothesis:** if these three signals really do warn us ahead of time, the year right before a real restriction should look different from a normal year.


In [ ]:
from scipy import stats
for col in ["Price_YoY_pct","Quantity_YoY_pct","Production_YoY_pct"]:
    g0 = reliable[reliable["Restriction_Next_Year"]==0][col].dropna()
    g1 = reliable[reliable["Restriction_Next_Year"]==1][col].dropna()
    u,p = stats.mannwhitneyu(g1,g0,alternative="two-sided")
    print(f"{col}: normal year median={g0.median():.2f}  year-before-restriction median={g1.median():.2f}  p={p:.4f}")


**What this tells us:** pooled across every country together, none of the three signals show a real difference. This does not mean the idea is wrong, it means lumping every country into one average hides what is really going on. The next step checks each country on its own.


---
## Step 4: Check each country on its own

**Why do this at all:** countries can behave in completely different ways for completely different reasons. Averaging them together can cancel out a real pattern in one country with an opposite pattern in another.


In [ ]:
for country in sorted(reliable["Country"].unique()):
    sub = reliable[reliable["Country"]==country]
    counts = sub["Restriction_Next_Year"].value_counts().to_dict()
    print(f"{country} (normal years={counts.get(0,0)}, year-before-restriction={counts.get(1,0)}):")
    print(sub.groupby("Restriction_Next_Year")[["Price_YoY_pct","Quantity_YoY_pct","Production_YoY_pct"]].median())
    print()


**India: the one clean, trustworthy pattern.** Both price and quantity are clearly higher in the year before a restriction than in a normal year. This matches the hypothesis exactly, rising price and rising selling activity, right before India pulls back. This is the one country where the story holds together end to end.

**Egypt did not show this pattern, and here is a real, plausible reason why.** Egypt's price does rise before a restriction, but its quantity actually falls, the opposite of the hypothesis. What lines up cleanly for Egypt instead is production, it drops before a restriction. This points to a different real story, Egypt's restrictions look like a reaction to a bad harvest at home, not to a rush of selling abroad. A supply problem, not a demand problem.

**Vietnam also did not pass, for a different, real reason.** Vietnam's quantity does rise before a restriction, matching the hypothesis, but its price actually falls, the opposite direction. One honest explanation, Vietnam's rice exports are heavily managed by state trading arrangements and government-set contracts, so its export price may not float freely the way a purely open market's would. The quantity signal still shows something real happening, the price signal does not tell the same clean story.

**Argentina and Russia cannot be judged fairly at all, for a simple reason.** Argentina only has 1 real restriction in this window, Russia only has 2. There are not enough real examples to say anything meaningful about a pattern for either of them, so we do not force a conclusion where the data cannot support one.


---
## Step 5: Build the actual model

**Why we still use all 5 countries together, even though only India's pattern is clean.** A model does not need every country to behave identically, it needs to know WHICH country it is looking at, so it can judge each one against its own normal behavior, not a blended average. We test this directly below.


In [ ]:
data = reliable.dropna(subset=["Price_YoY_pct","Quantity_YoY_pct","Production_YoY_pct"]).copy()
X_numeric = data[["Price_YoY_pct","Quantity_YoY_pct","Production_YoY_pct"]].values
y = data["Restriction_Next_Year"].values
country_dummies = pd.get_dummies(data["Country"], prefix="Country").values

def run_loocv(X, y):
    loo = LeaveOneOut()
    probs = []
    for train_idx, test_idx in loo.split(X):
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X[train_idx])
        X_test = scaler.transform(X[test_idx])
        model = LogisticRegression(class_weight="balanced", max_iter=1000)
        model.fit(X_train, y[train_idx])
        probs.append(model.predict_proba(X_test)[0][1])
    return np.array(probs)

probs_no_country = run_loocv(X_numeric, y)
probs_with_country = run_loocv(np.hstack([X_numeric, country_dummies]), y)
print("Without knowing the country: AUC =", round(roc_auc_score(y, probs_no_country),3))
print("Telling the model which country: AUC =", round(roc_auc_score(y, probs_with_country),3))


**What this tells us:** simply telling the model which country each row belongs to takes it from no-better-than-guessing (0.53) to a real, working result (0.68). This confirms what Step 4 already suggested, country identity carries real information the raw numbers alone do not.


## Step 6: Does using ONLY India, our cleanest case, do even better?

A fair question, since India is the one country with a genuinely clean pattern, would the model do best by ignoring everyone else and focusing on India alone?


In [ ]:
india_data = data[data["Country"]=="India"]
Xi = india_data[["Price_YoY_pct","Quantity_YoY_pct","Production_YoY_pct"]].values
yi = india_data["Restriction_Next_Year"].values
probs_india_only = run_loocv(Xi, yi)
print("India only: AUC =", round(roc_auc_score(yi, probs_india_only),3))


**A real, honest surprise: no, it does worse (0.37), not better.** With only India's 42 rows to learn from, the model does not have enough real examples on its own to learn a reliable rule, even though India's average pattern is genuinely real. The full 5-country model works better because it has more total examples to learn from, even though only one of those countries shows a clean individual pattern. This is worth remembering, more data across similar-but-not-identical situations can beat a smaller amount of perfectly clean data.


---
## Step 7: What else did we try, to see if the model could be improved?

We tested several other ideas on top of the 3 core signals, to see if any of them made the model better. None of them did. We are not running the code for all of these again here, since it would make this notebook harder to follow, but here is a plain, honest record of what was tried and what happened, so nothing is hidden.

| Idea tested | What it was | What happened |
|---|---|---|
| Contagion count | How many OTHER countries restricted the same year | Made the model worse |
| Price percentile | How high a country's price is compared to its own history | Made the model worse |
| Per-country standardizing | Adjusting each country's numbers to its own normal range | No improvement |
| Composite score | Combining price and quantity into one single number | No improvement |
| Domestic food inflation (CPI) | Swapping export price for the price people pay at home | Essentially tied, no real gain |
| Global rice price benchmark | Adding the world price on top of the country's own price | Made the model worse |
| Price gap | The difference between a country's own price and the world price | Made the model worse |
| Domestic food supply | How much rice is available at home after trade | Made the model worse, and had far less data available |

**Why this list matters, even though nothing on it worked:** it shows the 3-signal model was not left un-improved by accident, real attempts were made to beat it, and none succeeded. A number that survives 8 honest attempts to improve it is more trustworthy than one nobody ever tried to challenge.


---
## The most important takeaways from this notebook, to carry forward

1. **Price, Quantity, and Production were chosen because each one reflects a different real part of the story: the buyer-facing price, the country's own selling behavior, and the underlying harvest.**
2. **Pooled across all countries, none of the three signals show a real pattern. Checked one country at a time, India shows a genuinely clean, consistent pattern, both price and quantity rise before it restricts.**
3. **Egypt and Vietnam do not match the hypothesis, for real, different reasons: Egypt's story is about a shrinking harvest, not a rush of selling. Vietnam's export price may not move freely, since much of its rice trade is state-managed.**
4. **Argentina and Russia simply do not have enough real restriction events to judge fairly.**
5. **Telling the model which country it's looking at is what makes it work (AUC 0.53 to 0.68), not needing every country to share the same clean pattern.**
6. **India alone is not enough to build a good model from, even though its pattern is the cleanest. More total data, even from countries with messier patterns, produces a better result.**
7. **8 additional ideas were tested to improve the model further. None worked. This is treated as a real, valuable finding, not a failure, since it shows the 3-signal, multi-country model is a genuinely solid stopping point, not an accident.**

**This notebook produces the validated Rice export-side model.**


---
# PART 2: THE SAME QUESTION, FOR WHEAT

Everything below repeats the exact same method used for Rice, on Wheat instead. The reasoning for Price, Quantity, and Production is identical to Part 1, so it is not repeated here, only what is different or new is explained.

**Upload:** `comtrade_2002-2023_wheat.csv`, `comtrade_2014-2025_wheat.csv`


In [ ]:
uploaded = files.upload()

In [ ]:
wheat_events = events[events["CommodityClass_Name"]=="Wheat"].copy()
wheat_relevant_countries = sorted(wheat_events["Country_Name"].unique())
print("Countries with a real Wheat restriction history:", wheat_relevant_countries)


**A real difference worth noting straight away:** the Wheat country list is not the same as Rice's. Vietnam, which restricts Rice, never restricts Wheat. Ukraine and Kazakhstan restrict Wheat but never Rice. This confirms, again, that country scope must always come from the real events, never assumed to be the same across crops.


## Build the trade and production data

In [ ]:
w1 = pd.read_csv("comtrade_2002-2023_wheat.csv", index_col=False)
w2 = pd.read_csv("comtrade_2014-2025_wheat.csv", index_col=False)
comtrade_wheat = pd.concat([w1,w2], ignore_index=True)
comtrade_wheat = comtrade_wheat[comtrade_wheat["flowDesc"]=="Export"]
trade_wheat = comtrade_wheat.groupby(["reporterDesc","refYear"], as_index=False).agg(
    Export_Quantity_t=("qty", lambda x: x.sum()/1000), Export_Value_USD=("primaryValue","sum"))
trade_wheat = trade_wheat.rename(columns={"reporterDesc":"Country","refYear":"Year"})
trade_wheat = trade_wheat[trade_wheat["Country"].isin(wheat_relevant_countries)].copy()

zero_check = trade_wheat[trade_wheat["Export_Quantity_t"]==0]
print("Rows with zero quantity but real value:", len(zero_check))
print(zero_check["Country"].value_counts())


**A much bigger version of the same problem found in Rice.** 11 of Egypt's 24 years, nearly half, have no reported quantity at all. Egypt is fundamentally a wheat IMPORTER, the world's largest, not an exporter, so its export records are thin and prone to exactly this kind of gap. Treated the same way as before, as missing, not zero.


In [ ]:
trade_wheat["Export_Quantity_t"] = trade_wheat["Export_Quantity_t"].replace(0, np.nan)
trade_wheat["Unit_Price_USD_per_t"] = trade_wheat["Export_Value_USD"] / trade_wheat["Export_Quantity_t"]
trade_wheat = trade_wheat.replace([np.inf,-np.inf], np.nan)

prod_wheat = production_raw[(production_raw["Element"]=="Production") & (production_raw["Item"]=="Wheat")][["Area","Year","Value"]].rename(columns={"Area":"Country","Value":"Production_t"})


In [ ]:
full_wheat = pd.merge(trade_wheat, prod_wheat, on=["Country","Year"], how="inner")
full_wheat = full_wheat.sort_values(["Country","Year"]).reset_index(drop=True)

full_wheat["Prev_Year"] = full_wheat.groupby("Country")["Year"].shift(1)
full_wheat["YoY_Reliable"] = (full_wheat["Year"]-full_wheat["Prev_Year"])==1
full_wheat.loc[full_wheat["Prev_Year"].isna(),"YoY_Reliable"]=False
full_wheat["Price_YoY_pct"] = full_wheat.groupby("Country")["Unit_Price_USD_per_t"].pct_change(fill_method=None)*100
full_wheat["Quantity_YoY_pct"] = full_wheat.groupby("Country")["Export_Quantity_t"].pct_change(fill_method=None)*100
full_wheat["Production_YoY_pct"] = full_wheat.groupby("Country")["Production_t"].pct_change(fill_method=None)*100
for col in ["Price_YoY_pct","Quantity_YoY_pct","Production_YoY_pct"]:
    full_wheat.loc[~full_wheat["YoY_Reliable"], col] = None

ban_lookup_wheat = set(zip(wheat_events["Country_Name"], wheat_events["Start_Year"]))
full_wheat["Restriction_Next_Year"] = full_wheat.apply(lambda r: 1 if (r["Country"], r["Year"]+1) in ban_lookup_wheat else 0, axis=1)
reliable_wheat = full_wheat[full_wheat["YoY_Reliable"]==True].copy()
print("Reliable Wheat rows:", len(reliable_wheat))


## Test the hypothesis, pooled

In [ ]:
for col in ["Price_YoY_pct","Quantity_YoY_pct","Production_YoY_pct"]:
    g0 = reliable_wheat[reliable_wheat["Restriction_Next_Year"]==0][col].dropna()
    g1 = reliable_wheat[reliable_wheat["Restriction_Next_Year"]==1][col].dropna()
    u,p = stats.mannwhitneyu(g1,g0,alternative="two-sided")
    print(f"{col}: normal median={g0.median():.2f}  year-before-restriction median={g1.median():.2f}  p={p:.4f}")


**Same result as Rice: pooled together, none of the three signals show a real pattern.**


## Check each country on its own

In [ ]:
for country in sorted(reliable_wheat["Country"].unique()):
    sub = reliable_wheat[reliable_wheat["Country"]==country]
    counts = sub["Restriction_Next_Year"].value_counts().to_dict()
    print(f"{country} (normal={counts.get(0,0)}, year-before-restriction={counts.get(1,0)}):")
    print(sub.groupby("Restriction_Next_Year")[["Price_YoY_pct","Quantity_YoY_pct","Production_YoY_pct"]].median())
    print()


**India shows the same clean pattern here as it did for Rice, in a completely separate commodity and set of events.** Both price and quantity rise more before a restriction than in a normal year. This is real, independent confirmation, not a coincidence tied to one crop, India's behavior before a restriction looks consistent across two different foods.

**Ukraine has the best-balanced real sample of any country in this whole project, 11 normal years against 11 pre-restriction years.** Its quantity does rise before a restriction, matching the hypothesis, but its price actually falls, the opposite direction. A real, unresolved case, not enough evidence either way to call it a clean pass or a clean fail.

**Russia's story looks like a supply-shock, similar to Egypt's Rice pattern.** Its production drops sharply before a restriction, and its quantity falls too, this looks like a reaction to a bad harvest, not a rush of selling abroad.

**Egypt only has 1 real pre-restriction year for Wheat, and Kazakhstan only has 2.** Neither can be judged fairly, the same honest limit we hit with Argentina and Russia on the Rice side.


## Build the Wheat model

In [ ]:
data_wheat = reliable_wheat.dropna(subset=["Price_YoY_pct","Quantity_YoY_pct","Production_YoY_pct"]).copy()
Xw = data_wheat[["Price_YoY_pct","Quantity_YoY_pct","Production_YoY_pct"]].values
yw = data_wheat["Restriction_Next_Year"].values
cd_w = pd.get_dummies(data_wheat["Country"], prefix="Country").values

probs_w_no = run_loocv(Xw, yw)
probs_w_with = run_loocv(np.hstack([Xw, cd_w]), yw)
print("Wheat, without Country: AUC =", round(roc_auc_score(yw, probs_w_no),3))
print("Wheat, with Country:    AUC =", round(roc_auc_score(yw, probs_w_with),3))


**Same real lift as Rice, telling the model which country it's looking at genuinely helps.** The final number is somewhat lower than Rice's, which lines up with Wheat's messier underlying data, Egypt's missing-quantity problem in particular.


---
## The most important Wheat-specific takeaways, added to what Rice already taught us

1. **The country list is genuinely different for Wheat, Vietnam never restricts it, Ukraine and Kazakhstan only restrict Wheat, never Rice.**
2. **India's pattern holds up again, independently, in a second commodity, this is the strongest single piece of evidence in the whole project.**
3. **Ukraine gives us the best-balanced real sample anywhere in this project, but its own pattern is genuinely mixed, a real, honest result, not a clean pass.**
4. **Russia's story mirrors Egypt's Rice story, a supply-shock signal (falling production), not a demand-side one.**
5. **Adding Country still helps the model, the same real mechanism as Rice, even though the final Wheat number is a bit lower, reflecting messier underlying data.**
